In [ ]:
from resnet_utils import (
    plot_random_cluster_samples, 
    plot_random_class_samples, 
    generate_data_representations,
    load_data_model,
    run_deepdream
)

resnet18, val_loader, ds, device = load_data_model()
inputs, features, preds, labels = generate_data_representations(resnet18, val_loader, feat_name = 'avgpool', device=device)

In [ ]:
from local_corex import partition_data
from local_corex.nn_utils import compute_cluster_accuracies
from local_corex.nn_plotting import plot_perturved_accuracy_resnet, plot_logit_effects
import pickle
from_scratch = False

if from_scratch:
    print('computing clusters from scratch')
    num_clusters=100
    indexes = partition_data(features, n_partitions=num_clusters, phate_dim=10, n_jobs=-2, seed=42)
    print('saving clusters')
    with open('resnet_100_indexes.pkl', 'wb') as f:
        pickle.dump(indexes, f)
else:
    print("loading clusters")
    with open('resnet_100_indexes.pkl', 'rb') as f:
        indexes = pickle.load(f)

In [ ]:
import numpy as np
from collections import defaultdict
import torch

if isinstance(labels, torch.Tensor):
    labels = labels.numpy()
cluster_counts = defaultdict(dict)

for cluster_id, mask in enumerate(indexes):
    cluster_labels = labels[mask]
    unique, counts = np.unique(cluster_labels, return_counts=True)
    cluster_counts[cluster_id] = dict(zip(unique, counts))

for cluster_id, counts in cluster_counts.items():
    sorted_counts = sorted(counts.items(), key=lambda x: -x[1])
    total = sum(counts.values())
    # percentages = sorted_counts
    percentages = {k: f"{(v/total)*100:.1f}%" for k, v in sorted_counts}
    print(f"Cluster {cluster_id} ({total} samples):")
    print("  Class distribution:", percentages)

# Looking at Partition 58

In [ ]:
from local_corex import LinearCorex

partition=58
x = np.concatenate([features[indexes[partition]], preds[indexes[partition]]], axis=1)

corex_58 = LinearCorex(20, seed=42)
corex_58.fit(x)
print(corex_58.tcs)

In [ ]:
import torchvision.models as models

weights = models.ResNet18_Weights.DEFAULT
class_names = weights.meta['categories']

plot_random_cluster_samples(ds, indexes, 58, class_names, sample_count=12, seed=124)

## Factor 0

In [ ]:
base_accuracies, base_probs = compute_cluster_accuracies(
    resnet18, val_loader, device, indexes, return_probs=True
)
diff_probs = plot_perturved_accuracy_resnet(
    resnet18, 
    corex_58, 
    val_loader, 
    indexes,
    device, 
    base_accuracies=base_accuracies, 
    base_probs=base_probs, 
    factor_num=0, 
    num_drop=100,
    hidden_dim=512
)

In [ ]:
from deepdream import DeepDreamer, TargetSpec, dream_images_and_plot

top_k_ind = np.argsort(corex_58.mis[0][:512])[::-1][:50].astype(np.int64).copy()
top_k_ind = top_k_ind.tolist()
top_k_weights = corex_58.mis[0][:512][top_k_ind].tolist()

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def preprocess(x):
    return (x - mean.to(x.device)) / std.to(x.device)

dreamer = DeepDreamer(
    resnet18,
    preprocess=preprocess,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

run_deepdream(
    corex_58, 
    dreamer,
    ds,
    steps=25,
    step_size=0.02,
    jitter=16,
    octaves=4,
    octave_scale=1.4,
    show_original_above=True,
    factor_num=0, 
    latent_dim=512, 
    node_num=100, 
    image_ids=[3, 10, 25, 40]
)


In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[indexes[partition]], dim=0), 
    class_names, 
    bottom_vals=2,
    top_vals=8
)

In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[indexes[28]], dim=0), 
    class_names, 
    bottom_vals=5,
    top_vals=5
)

Look at how if you were to subtract off the bird features you would most likely notice the stripe pattern of a zebra

In [ ]:
plot_random_class_samples(ds, class_label=83, class_names=class_names, sample_count=4, seed=124)
plot_random_class_samples(ds, class_label=138, class_names=class_names, sample_count=4, seed=124)

In [ ]:
for i, name in enumerate(class_names):
    if name == "bustard":
        print(i)

## Factor 1

In [ ]:
base_accuracies, base_probs = compute_cluster_accuracies(
    resnet18, val_loader, device, indexes, return_probs=True
)
diff_probs = plot_perturved_accuracy_resnet(
    resnet18, 
    corex_58, 
    val_loader, 
    indexes,
    device, 
    base_accuracies=base_accuracies, 
    base_probs=base_probs, 
    factor_num=1, 
    num_drop=100,
    hidden_dim=512
)

In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[indexes[partition]], dim=0), 
    class_names, 
    bottom_vals=2,
    top_vals=8
)

In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[indexes[21]], dim=0), 
    class_names, 
    bottom_vals=5,
    top_vals=5
)

Notice the classes affected have a net or white line across. The African Grey is often pictured with a perch or rope that has similar features.

In [ ]:
plot_random_cluster_samples(ds, indexes, 21, class_names, sample_count=12, seed=18)

In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[~indexes[partition]], dim=0), 
    class_names, 
    bottom_vals=5, 
    top_vals=5
)

In [ ]:
run_deepdream(
    corex_58, 
    dreamer,
    ds,
    steps=25,
    step_size=0.02,
    jitter=16,
    octaves=4,
    octave_scale=1.4,
    show_original_above=True,
    factor_num=1, 
    latent_dim=512, 
    node_num=100, 
    image_ids=[3, 10, 25, 40]
)

# Partition 78

In [ ]:
# from local_corex.latent_transformer import CorExWrapper

partition=78
x = np.concatenate([features[indexes[partition]], preds[indexes[partition]]], axis=1)

corex_78 = LinearCorex(20, seed=42)
corex_78.fit(x)
print(corex_78.tcs)

In [ ]:
plot_random_cluster_samples(ds, indexes, partition, class_names, sample_count=12, seed=148)

## Factor 0

In [ ]:
diff_probs = plot_perturved_accuracy_resnet(
    resnet18, 
    corex_78, 
    val_loader, 
    indexes,
    device, 
    base_accuracies=base_accuracies, 
    base_probs=base_probs, 
    factor_num=0, 
    num_drop=100,
    hidden_dim=512
)

In [ ]:
run_deepdream(
    corex_78, 
    dreamer,
    ds,
    steps=25,
    step_size=0.02,
    jitter=16,
    octaves=4,
    octave_scale=1.4,
    show_original_above=True,
    factor_num=0, 
    latent_dim=512, 
    node_num=100, 
    image_ids=[3, 10, 25, 40]
)

In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[indexes[partition]], dim=0), 
    class_names, 
    bottom_vals=2,
    top_vals=8
)

In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[indexes[7]], dim=0), 
    class_names, 
    bottom_vals=5,
    top_vals=5
)

I think this feature was used to find the stem of the fruit/flower and can look a bit like the ropes/leather of a muzzle/cart/plow

In [ ]:
plot_random_cluster_samples(ds, indexes, 7, class_names, sample_count=36, seed=148)

## Factor 1

In [ ]:
diff_probs = plot_perturved_accuracy_resnet(
    resnet18, 
    corex_78, 
    val_loader, 
    indexes,
    device, 
    base_accuracies=base_accuracies, 
    base_probs=base_probs, 
    factor_num=1, 
    num_drop=100,
    hidden_dim=512
)

In [ ]:
run_deepdream(
    corex_78, 
    dreamer,
    ds,
    steps=25,
    step_size=0.02,
    jitter=16,
    octaves=4,
    octave_scale=1.4,
    show_original_above=True,
    factor_num=1, 
    latent_dim=512, 
    node_num=100, 
    image_ids=[3, 10, 25, 40]
)

In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[indexes[partition]], dim=0), 
    class_names, 
    bottom_vals=2,
    top_vals=8
)

In [ ]:
plot_logit_effects(
    torch.mean(diff_probs[indexes[15]], dim=0), 
    class_names, 
    bottom_vals=5,
    top_vals=5
)

My guess is that this feature relates to the tail of the stingray, the tubes of the scuba diver, the tentacles of the jellyfish, and the actual part of the snorkal. This could look like a stem or vine connecting the plant.

In [ ]:
plot_random_cluster_samples(ds, indexes, 15, class_names, sample_count=12, seed=125)